In [1]:
import pandas as pd
import numpy as np

# Input dataset path (update path if needed)
input_path = "housing_price_dataset.csv"
df = pd.read_csv(input_path)

# Show first few rows
df.head()


,SquareFeet,Bedrooms,Bathrooms,Neighborhood,YearBuilt,Price
0,2126,4,1,Rural,1969,215355.283618
1,2459,3,2,Rural,1980,195014.221626
2,1860,2,1,Suburb,1970,306891.012076
3,2294,2,1,Urban,1996,206786.787153
4,2130,5,2,Suburb,2001,272436.239065


In [2]:
# Normalize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

# Helper function to find matching columns
def find_col(keys):
    for k in keys:
        for c in df.columns:
            if k in c:
                return c
    return None

price_col = find_col(['price', 'amount', 'rent'])
size_col = find_col(['size', 'sqft', 'square', 'area'])
bed_col = find_col(['bed', 'bedrooms', 'beds'])
bath_col = find_col(['bath', 'bathrooms'])
loc_col = find_col(['location', 'neighborhood', 'area', 'district'])
ptype_col = find_col(['property_type','type'])
dev_col = find_col(['developer','builder','developer_name'])
year_col = find_col(['year_built','year','built'])
furn_col = find_col(['furnishing','furnishing_status','furnished'])

print("Price:", price_col)
print("Size:", size_col)
print("Bedrooms:", bed_col)
print("Bathrooms:", bath_col)
print("Location:", loc_col)
print("Property type:", ptype_col)
print("Developer:", dev_col)
print("Year Built:", year_col)
print("Furnishing:", furn_col)


Price: price
Size: squarefeet
Bedrooms: bedrooms
Bathrooms: bathrooms
Location: neighborhood
Property type: None
Developer: None
Year Built: yearbuilt
Furnishing: None


In [3]:
# Convert numeric columns
def to_numeric_series(s):
    s = s.astype(str).str.replace(r'[^0-9.\-]', '', regex=True)
    s = s.replace(['', 'nan', 'None', 'None.'], np.nan)
    return pd.to_numeric(s, errors='coerce')

if price_col: df['price'] = to_numeric_series(df[price_col])
if size_col: df['size_sqft'] = to_numeric_series(df[size_col])
if bed_col: df['bedrooms'] = pd.to_numeric(df[bed_col], errors='coerce')
if bath_col: df['bathrooms'] = pd.to_numeric(df[bath_col], errors='coerce')

# Standardize text fields
if loc_col: df['location'] = df[loc_col].astype(str).str.strip()
if ptype_col: df['property_type'] = df[ptype_col].astype(str).str.strip()
if dev_col: df['developer'] = df[dev_col].astype(str).str.strip()
if furn_col:
    furn = df[furn_col].astype(str).str.lower()
    df['furnishing_status'] = np.where(furn.str.contains('furn', na=False), 'Furnished',
                              np.where(furn.str.contains('unfurn|not fur|un-fur|not furnished', na=False), 'Unfurnished',
                              np.where(furn.str.contains('part', na=False), 'Part-Furnished', 'Other')))


In [4]:
# Remove duplicates
df = df.drop_duplicates()

# Remove rows with invalid price/size
df = df[(df['price'] > 0) & (df['size_sqft'] > 0)]

# Fill missing bedrooms/bathrooms with median
if df['bedrooms'].isna().any():
    df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median()).astype(int)
if df['bathrooms'].isna().any():
    df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].median()).astype(int)


In [5]:
# Price per sqft
df['price_per_sqft'] = df['price'] / df['size_sqft']

# Property Age
if year_col:
    df['year_built'] = pd.to_numeric(df[year_col], errors='coerce')
    df['property_age'] = 2025 - df['year_built']

# Listing Category based on price quantiles
try:
    df['listing_category'] = pd.qcut(df['price'], q=3, labels=['Budget','Mid-Range','High-End'])
except:
    bins = np.quantile(df['price'].dropna(), [0, 0.33, 0.66, 1.0])
    df['listing_category'] = pd.cut(df['price'], bins=bins, labels=['Budget','Mid-Range','High-End'], include_lowest=True)

df.head()


,squarefeet,bedrooms,bathrooms,neighborhood,yearbuilt,price,size_sqft,location,price_per_sqft,year_built,property_age,listing_category
0,2126,4,1,Rural,1969,215355.283618,2126,Rural,101.295994,1969,56,Mid-Range
1,2459,3,2,Rural,1980,195014.221626,2459,Rural,79.306312,1980,45,Mid-Range
2,1860,2,1,Suburb,1970,306891.012076,1860,Suburb,164.995168,1970,55,High-End
3,2294,2,1,Urban,1996,206786.787153,2294,Urban,90.142453,1996,29,Mid-Range
4,2130,5,2,Suburb,2001,272436.239065,2130,Suburb,127.904338,2001,24,High-End


In [6]:
total_listings = len(df)
avg_price = df['price'].mean()
avg_size = df['size_sqft'].mean()
avg_pps = df['price_per_sqft'].mean()
highest_price = df['price'].max()

print("Total Listings:", total_listings)
print("Average Price:", avg_price)
print("Average Size:", avg_size)
print("Average Price/Sqft:", avg_pps)
print("Highest Price:", highest_price)


Total Listings: 49978
Average Price: 224931.66795958765
Average Size: 2006.752551122494
Average Price/Sqft: 113.35975370375405
Highest Price: 492195.2599720151


In [7]:
# Save cleaned data
output_path = "dubai_housing_enriched.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned & enriched dataset saved to {output_path}")


Cleaned & enriched dataset saved to dubai_housing_enriched.csv
